# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer, make_column_selector as selector
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    BaggingClassifier,
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
)

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [3]:
from sklearn.compose import ColumnTransformer, make_column_selector as selector
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Separate target
y = spaceship["Transported"]
X = spaceship.drop(columns=["Transported", "PassengerId", "Name"], errors="ignore")

# Selectors
num_sel = selector(dtype_include=["number"])
cat_sel = selector(dtype_include=["object", "category", "bool"])

# Numeric pipeline
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical pipeline
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine
preprocess = ColumnTransformer([
    ("num", num_pipe, num_sel),
    ("cat", cat_pipe, cat_sel)
])



**Perform Train Test Split**

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [5]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

bag_pipe = Pipeline([
    ("prep", preprocess),  # ColumnTransformer pipeline from earlier
    ("clf", BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=42),
        n_estimators=200,
        bootstrap=True,      # bagging; set False for pasting
        n_jobs=-1,
        random_state=42
    ))
])

bag_pipe.fit(X_train, y_train)
y_pred_bag = bag_pipe.predict(X_test)
print("Bagging Accuracy:", accuracy_score(y_test, y_pred_bag))





Bagging Accuracy: 0.7975848188614146


- Random Forests

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

rf_pipe = Pipeline([
    ("prep", preprocess),  # same ColumnTransformer as before
    ("clf", RandomForestClassifier(
        n_estimators=200,   # number of trees
        max_depth=None,     # let trees expand fully
        n_jobs=-1,          # use all cores
        random_state=42
    ))
])

rf_pipe.fit(X_train, y_train)
y_pred_rf = rf_pipe.predict(X_test)
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))



Random Forest Accuracy: 0.7889591719378953


- Gradient Boosting

In [7]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

gb_pipe = Pipeline([
    ("prep", preprocess),  # same ColumnTransformer
    ("clf", GradientBoostingClassifier(
        n_estimators=200,     # number of boosting stages
        learning_rate=0.1,    # shrinkage parameter
        max_depth=3,          # depth of individual trees
        random_state=42
    ))
])

gb_pipe.fit(X_train, y_train)
y_pred_gb = gb_pipe.predict(X_test)
print("Gradient Boosting Accuracy:", accuracy_score(y_test, y_pred_gb))



Gradient Boosting Accuracy: 0.7981598619896493


- Adaptive Boosting

In [8]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

ada_pipe = Pipeline([
    ("prep", preprocess),  # your ColumnTransformer from earlier
    ("clf", AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1, random_state=42),  # decision stump
        n_estimators=300,
        learning_rate=0.1,
        random_state=42
    ))
])

ada_pipe.fit(X_train, y_train)
y_pred_ada = ada_pipe.predict(X_test)
print("AdaBoost Accuracy:", accuracy_score(y_test, y_pred_ada))



AdaBoost Accuracy: 0.7469810235767682


Which model is the best and why?

In [9]:
results = {
    "Bagging": accuracy_score(y_test, y_pred_bag),
    "Random Forest": accuracy_score(y_test, y_pred_rf),
    "Gradient Boosting": accuracy_score(y_test, y_pred_gb),
    "AdaBoost": accuracy_score(y_test, y_pred_ada),
}

print("Model Comparison:")
for model, acc in results.items():
    print(f"{model}: {acc:.4f}")

best_model = max(results, key=results.get)
print("\n✅ Best Model:", best_model, "with accuracy:", results[best_model])



Model Comparison:
Bagging: 0.7976
Random Forest: 0.7890
Gradient Boosting: 0.7982
AdaBoost: 0.7470

✅ Best Model: Gradient Boosting with accuracy: 0.7981598619896493


After comparing ensemble methods, Gradient Boosting achieved the highest accuracy (≈79.8%).
Bagging and Random Forest were close, but slightly behind. AdaBoost had the lowest performance.
The reason Gradient Boosting is best here is that it builds trees sequentially, learning from the errors of previous trees, which helps capture subtle patterns in the dataset. Random Forest and Bagging reduce variance effectively, but lack this sequential correction, which explains their slightly lower performance.